# Lab agent : Assistant Ndimbal
Deux sections : (1) un agent **sans mémoire**, (2) le même agent **avec mémoire de session**.
Toutes les données sont fictives.

## Préparation
On charge la clé depuis `.env` (jamais affichée) et on importe Google ADK.

In [27]:
import os, uuid
from dotenv import load_dotenv

load_dotenv()
# ADK / google-genai lit GOOGLE_API_KEY ; on accepte aussi GEMINI_API_KEY.
if not os.getenv("GOOGLE_API_KEY") and os.getenv("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("Clé détectée :", bool(os.getenv("GOOGLE_API_KEY")))

Clé détectée : True


In [28]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MODEL = "gemini-3.5-flash-lite"
APP_NAME = "ndimbal_lab"
USER_ID = "mame_diarra"

## Définition de l'agent
Une seule définition, réutilisée dans les deux sections. Les consignes décrivent les canaux de rappel et les moyens de paiement.

In [29]:
INSTRUCTION = """Tu es « Assistant Ndimbal », assistant d'un assureur sénégalais (données fictives).
Tu réponds en français, de façon courte et claire, aux questions d'un assuré sur les rappels de prime.
- Canaux de rappel possibles : SMS, appel, WhatsApp, conseiller.
- Moyens de paiement : Wave, Orange Money, agence.
Ne donne jamais de numéro de contrat ni de montant : invite l'assuré à contacter son conseiller.
Si tu ne connais pas une préférence de l'assuré, dis-le simplement et propose les canaux disponibles."""

def creer_agent():
    return Agent(
        name="assistant_ndimbal",
        model=MODEL,
        description="Répond aux questions sur les rappels de prime.",
        instruction=INSTRUCTION,
    )

## Fonction d'échange (async)
`demander` envoie un message au runner et renvoie la réponse finale de l'agent.

In [30]:
async def demander(runner, session_id, texte):
    message = types.Content(role="user", parts=[types.Part(text=texte)])
    reponse = ""
    async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            reponse = "".join(p.text or "" for p in event.content.parts)
    print(f"Assuré : {texte}\nNdimbal : {reponse}\n")
    return reponse

# Section 1 : agent SANS mémoire
Chaque message est envoyé dans une **nouvelle session** : l'agent ne garde rien d'un échange à l'autre.

In [31]:
service_1 = InMemorySessionService()
runner_1 = Runner(agent=creer_agent(), app_name=APP_NAME, session_service=service_1)

async def nouvelle_session(service):
    sid = str(uuid.uuid4())
    await service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=sid)
    return sid

Question de base sur les rappels de prime.

In [32]:
await demander(runner_1, await nouvelle_session(service_1),
               "Comment puis-je être rappelé pour ma prime et comment payer ?")

Assuré : Comment puis-je être rappelé pour ma prime et comment payer ?
Ndimbal : Bonjour ! Je suis l'Assistant Ndimbal. 

Pour vos rappels de prime, nous utilisons les canaux suivants : SMS, appel téléphonique, WhatsApp ou votre conseiller dédié. 

Concernant les paiements, vous pouvez régler via **Wave**, **Orange Money** ou directement en **agence**.

Pour connaître votre montant exact et votre numéro de contrat, je vous invite à **contacter directement votre conseiller**.



"Bonjour ! Je suis l'Assistant Ndimbal. \n\nPour vos rappels de prime, nous utilisons les canaux suivants : SMS, appel téléphonique, WhatsApp ou votre conseiller dédié. \n\nConcernant les paiements, vous pouvez régler via **Wave**, **Orange Money** ou directement en **agence**.\n\nPour connaître votre montant exact et votre numéro de contrat, je vous invite à **contacter directement votre conseiller**."

Premier message : l'assurée se présente et donne sa préférence.

In [33]:
await demander(runner_1, await nouvelle_session(service_1),
               "Je m'appelle Mame Diarra, j'ai un contrat Études et je préfère WhatsApp")

Assuré : Je m'appelle Mame Diarra, j'ai un contrat Études et je préfère WhatsApp
Ndimbal : Bonjour Mame Diarra. C'est bien noté pour votre préférence via WhatsApp. 

Pour tout ce qui concerne les détails de votre contrat Études ou le montant exact de votre prime, je vous invite à contacter directement votre conseiller. 

Restant à votre disposition !



"Bonjour Mame Diarra. C'est bien noté pour votre préférence via WhatsApp. \n\nPour tout ce qui concerne les détails de votre contrat Études ou le montant exact de votre prime, je vous invite à contacter directement votre conseiller. \n\nRestant à votre disposition !"

Question de suivi dans une **autre session** : l'agent a oublié la préférence.

In [34]:
await demander(runner_1, await nouvelle_session(service_1),
               "Par quel canal dois-je recevoir mes rappels ?")

Assuré : Par quel canal dois-je recevoir mes rappels ?
Ndimbal : Je ne connais pas encore votre préférence pour les rappels de prime. 

Vous pouvez les recevoir par SMS, appel, WhatsApp ou via votre conseiller. 

Pour mettre à jour votre préférence ou pour toute question sur votre contrat, veuillez contacter votre conseiller.



'Je ne connais pas encore votre préférence pour les rappels de prime. \n\nVous pouvez les recevoir par SMS, appel, WhatsApp ou via votre conseiller. \n\nPour mettre à jour votre préférence ou pour toute question sur votre contrat, veuillez contacter votre conseiller.'

# Section 2 : agent AVEC mémoire de session
Même agent, mais les deux messages partagent **la même session** : l'historique est conservé par le `SessionService`.

In [35]:
service_2 = InMemorySessionService()
runner_2 = Runner(agent=creer_agent(), app_name=APP_NAME, session_service=service_2)
session_id = await nouvelle_session(service_2)

Même premier message, dans la session partagée.

In [36]:
await demander(runner_2, session_id,
               "Je m'appelle Mame Diarra, j'ai un contrat Études et je préfère WhatsApp")

Assuré : Je m'appelle Mame Diarra, j'ai un contrat Études et je préfère WhatsApp
Ndimbal : Bonjour Mame Diarra, c'est bien pris en compte ! 

Pour le paiement de votre prime, vous pouvez utiliser **Wave**, **Orange Money** ou passer en **agence**. 

Concernant le montant exact et votre numéro de contrat, je vous invite à contacter directement votre conseiller. Avez-vous besoin d'autre chose ?



"Bonjour Mame Diarra, c'est bien pris en compte ! \n\nPour le paiement de votre prime, vous pouvez utiliser **Wave**, **Orange Money** ou passer en **agence**. \n\nConcernant le montant exact et votre numéro de contrat, je vous invite à contacter directement votre conseiller. Avez-vous besoin d'autre chose ?"

Même question de suivi, **même session** : l'agent répond « WhatsApp ».

In [37]:
reponse = await demander(runner_2, session_id, "Par quel canal dois-je recevoir mes rappels ?")
assert "whatsapp" in reponse.lower(), "L'agent n'a pas retenu la préférence"

Assuré : Par quel canal dois-je recevoir mes rappels ?
Ndimbal : Vous préférez recevoir vos rappels par **WhatsApp**, comme vous me l'avez indiqué. 

Les autres canaux disponibles sont le **SMS**, l'**appel** téléphonique ou directement via votre **conseiller**.



**Conclusion** : sans session partagée, l'agent oublie ; avec la même session, il retient le nom, le contrat et le canal préféré.